In [1]:
import sys; sys.path.append("..")
import pandas as pd
from sklearn.metrics import mean_absolute_error
from src.features import buat_lag, tambah_fitur

df = pd.read_csv("../data/clean.csv", parse_dates=["waktu"])
df["target"] = df.groupby("stasiun")["pm25_ugm3"].shift(-3)

d = buat_lag(df).dropna(subset=["pm25_ugm3", "target"])
X = tambah_fitur(d)
tr = d["waktu"] < "2026-03-01"
va = (d["waktu"] >= "2026-03-01") & (d["waktu"] < "2026-05-01")   # test (>= Mei) tidak disentuh
yt, yv = d.loc[tr, "target"], d.loc[va, "target"]

print(tr.sum(), va.sum())   # harus 71777 dan 7238
print("MAE baseline:", round(mean_absolute_error(yv, d.loc[va, "pm25_ugm3"]), 3))

71777 7238
MAE baseline: 6.255


## Model

- Fitur (hanya informasi sampai jam saat ini): PM2.5 sekarang, lag 1/3/24 jam, tren 3 jam, jam (sin/cos), hari, cuaca, arah angin (sin/cos), stasiun
- Validation: baseline MAE 6,255 → Ridge 5,113 → HistGradientBoosting 4,695 (model terpilih)
- Batas alarm peringatan: 45 µg/m³ pada prediksi. Dipilih di validation karena F1 tertinggi (recall [0.465], precision [0.328]). Lebih rendah dari 55 karena ingin memperingatkan sebelum udara memburuk.
- Keterbatasan: hanya 43 jam kejadian >55 di validation, jadi angka recall/precision mudah bergoyang. Validation jatuh di musim hujan, test di awal kemarau.
- Test (Mei-Jun 2026) belum dibuka

In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

ridge = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge()).fit(X[tr], yt)
hgb = HistGradientBoostingRegressor(random_state=42).fit(X[tr], yt)

for nama, m in [("Ridge", ridge), ("HistGradientBoosting", hgb)]:
    print(nama, round(mean_absolute_error(yv, m.predict(X[va])), 3))

Ridge 5.113
HistGradientBoosting 4.695


In [3]:
p = hgb.predict(X[va]); y = yv > 55
hasil = []
for th in range(35, 61, 5):
    a = p > th
    rec, pre = (y & a).sum() / y.sum(), (y & a).sum() / max(a.sum(), 1)
    hasil.append((th, int(a.sum()), round(rec, 3), round(pre, 3), round(2*rec*pre / max(rec + pre, 1e-9), 3)))
pd.DataFrame(hasil, columns=["batas", "alarm", "recall", "precision", "f1"])

,batas,alarm,recall,precision,f1
0,35,360,0.884,0.106,0.189
1,40,145,0.651,0.193,0.298
2,45,61,0.465,0.328,0.385
3,50,24,0.209,0.375,0.269
4,55,7,0.093,0.571,0.160
5,60,4,0.093,1.000,0.170


In [4]:
BATAS_ALARM = 45   #F1 tertinggi
a = p > BATAS_ALARM
tp = (y & a).sum()
print(f"batas {BATAS_ALARM}: alarm {a.sum()}, recall {tp/y.sum():.3f}, precision {tp/a.sum():.3f}, kejadian >55: {y.sum()}")

batas 45: alarm 61, recall 0.465, precision 0.328, kejadian >55: 43


In [5]:
pd.DataFrame({"stasiun": d.loc[va, "stasiun"], "model": (yv - p).abs(), "baseline": (yv - d.loc[va, "pm25_ugm3"]).abs()}).groupby("stasiun").mean().round(2)

,model,baseline
stasiun,,
Antapani,4.39,5.86
Cibeureum,5.16,6.94
Dago,3.44,4.32
Soekarno-Hatta,6.35,8.87
Ujungberung,4.15,5.30
